# TDE-HMM — Análisis individual de experimento

In [ ]:
# Cambia solo esta línea
YAML_REL_PATH = "canonical/tde_validation_k2_t7.yaml"

import sys, warnings, json, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, yaml
from pathlib import Path
from collections import Counter
from scipy import stats
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120})

PROJECT_NAME = "eeg_hmm_plattform"
current = Path.cwd()
while current.name != PROJECT_NAME:
    if current.parent == current:
        raise RuntimeError(f"No se encontro {PROJECT_NAME}")
    current = current.parent
PROJECT_ROOT = current

yaml_path = PROJECT_ROOT / "configs" / "experiments" / YAML_REL_PATH
with open(yaml_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

PIPELINE_TYPE = cfg["experiment"].get("pipeline_type", "feature")
EXP_NAME      = cfg["experiment"]["name"]
K             = cfg["pipeline"]["hmm"]["k_states"]
COV_TYPE      = cfg["pipeline"]["hmm"]["covariance_type"]
PCA_VAR       = cfg["pipeline"]["pca"]["variance_retained"]

if PIPELINE_TYPE == "tde":
    N_LAGS   = cfg["tde"]["n_lags"]
    LAG_STEP = cfg["tde"].get("lag_step", 1)
    SFREQ    = cfg["tde"].get("sfreq", 250.0)
    STEP_MS  = 1000.0 / SFREQ
else:
    STEP_MS  = float(cfg.get("windowing", {}).get("step_ms", 100))
    N_LAGS   = None

STATE_COLORS = ["#E63946","#2196F3","#4CAF50","#FF9800","#9C27B0","#00BCD4"]
SAVE_FIGURES = True
FIG_DIR      = PROJECT_ROOT / "reports" / "figures" / "tde"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name):
    if SAVE_FIGURES:
        plt.savefig(FIG_DIR / f"{EXP_NAME}_{name}.png", dpi=150, bbox_inches="tight")

print(f"experimento : {EXP_NAME}")
print(f"pipeline    : {PIPELINE_TYPE}")
print(f"K={K} | cov={COV_TYPE} | pca={PCA_VAR}")
if PIPELINE_TYPE == "tde":
    print(f"n_lags={N_LAGS} | lag_step={LAG_STEP} | sfreq={SFREQ}Hz | step_ms={STEP_MS:.1f}ms")


---
## Carga del experimento

In [ ]:
output_dir = PROJECT_ROOT / cfg["paths"]["output_dir"].replace("../../","").replace("../","")
exp_dir    = output_dir / EXP_NAME

model     = joblib.load(exp_dir / f"hmm_model_k{K}.pkl")
viterbi   = np.load(exp_dir / f"viterbi_paths_k{K}.npy")
X_pca     = np.load(exp_dir / "X_pca.npy")
lengths   = np.load(exp_dir / "lengths.npy")
posterior = model.predict_proba(X_pca)
transmat  = model.transmat_

manifest  = json.loads((exp_dir / "feature_manifest.json").read_text(encoding="utf-8"))
summary   = json.loads((exp_dir / "decoding_summary.json").read_text(encoding="utf-8"))
df_fo     = pd.read_csv(exp_dir / "fo_by_subject.csv")
df_stats  = pd.read_csv(exp_dir / "state_stats.csv")

# subject_meta desde manifest
labels_csv = pd.read_csv(PROJECT_ROOT / "configs" / "subject_labels.csv")
labels = {}
for _, row in labels_csv.iterrows():
    base = row["npy_file"].split("_sin_contexto")[0].upper()
    labels[base] = {"condition": row["condition"], "group": row["group"]}

subject_meta = []
idx = 0
fif_files_ordered = list(manifest["fif_files"].keys())
n_lags   = manifest.get("n_lags", 0)
lag_step = manifest.get("lag_step", 1)
sfreq    = cfg.get("tde", {}).get("sfreq", 250.0) if PIPELINE_TYPE == "tde" else 250.0

for fif_name in fif_files_ordered:
    stem    = Path(fif_name).stem
    base_id = stem.split("_")[0].upper()
    match   = re.search(r"_(NOGO|GO)_", fif_name, re.IGNORECASE)
    row     = labels.get(base_id, {})
    cond    = match.group(1).upper() if match else row.get("condition", "UNKNOWN")
    group   = row.get("group", "UNKNOWN")

    # n_windows desde lengths usando la longitud modal por sesion
    n_trials_approx = sum(1 for l in lengths if l > 0)
    sub_lengths = []
    tmp = idx
    while tmp < len(lengths) and (not sub_lengths or lengths[tmp] == lengths[idx]):
        sub_lengths.append(lengths[tmp])
        tmp += 1
    n_win = sum(sub_lengths)

    subject_meta.append({
        "fname":      stem,
        "subject_id": base_id,
        "condition":  cond,
        "group":      group,
        "start":      idx,
        "end":        idx + n_win,
        "n_windows":  n_win,
    })
    idx += n_win

df_meta = pd.DataFrame(subject_meta)

print(f"X_pca   : {X_pca.shape}")
print(f"lengths : {lengths.shape} | suma={lengths.sum():,} | modal={np.bincount(lengths).argmax()}")
print(f"sesiones: {len(subject_meta)}")
print()
print(df_meta.groupby(["group","condition"]).size().to_string())


---
## Metricas por estado

In [ ]:
display(Markdown(f"### {EXP_NAME}"))
display(df_stats.style
    .background_gradient(subset=["FO_hard","FO_soft"], cmap="RdYlGn")
    .background_gradient(subset=["self_trans"], cmap="RdYlGn_r")
    .background_gradient(subset=["dwell_analytic_ms"], cmap="Blues")
    .background_gradient(subset=["confidence_mean"], cmap="Greens")
    .format({
        "FO_hard":"{:.4f}","FO_soft":"{:.4f}",
        "self_trans":"{:.4f}","dwell_analytic_ms":"{:.1f}",
        "dwell_median_ms":"{:.1f}","dwell_mean_ms":"{:.1f}",
        "confidence_mean":"{:.4f}","entropy_H":"{:.4f}",
        "n_windows":"{:,}","n_runs":"{:,}"
    })
)

print(f"
Score: {summary[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)]}/4 — {summary[chr(118)+chr(101)+chr(114)+chr(100)+chr(105)+chr(99)+chr(116)]}")
for k, v in summary["checks"].items():
    print(f"  {chr(79)+chr(75) if v else chr(70)+chr(65)+chr(73)+chr(76)} {k}")


---
## Matriz de transicion

In [ ]:
labels_s = [f"S{s}" for s in range(K)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.heatmap(transmat, ax=axes[0], annot=True, fmt=".4f", cmap="Blues",
            xticklabels=labels_s, yticklabels=labels_s, vmin=0, vmax=1, linewidths=0.8)
axes[0].set_title("Probabilidades de transicion", fontweight="bold")
axes[0].set_xlabel("Estado destino"); axes[0].set_ylabel("Estado origen")

off = transmat.copy(); np.fill_diagonal(off, 0)
rs  = off.sum(axis=1, keepdims=True)
off_n = np.divide(off, rs, out=np.zeros_like(off), where=rs!=0)
sns.heatmap(off_n, ax=axes[1], annot=True, fmt=".3f", cmap="Oranges",
            xticklabels=labels_s, yticklabels=labels_s, vmin=0, vmax=1, linewidths=0.8)
axes[1].set_title("Flujo entre estados (off-diagonal)", fontweight="bold")
axes[1].set_xlabel("Estado destino"); axes[1].set_ylabel("Estado origen")

plt.suptitle(f"Transiciones — {EXP_NAME}", fontsize=13, fontweight="bold")
plt.tight_layout(); save_fig("transmat"); plt.show()


---
## Variabilidad entre sujetos

In [ ]:
fo_wide = df_fo.pivot_table(index=["subject_id","condition","group"],
                             columns="state", values="FO").reset_index()

groups_present = df_fo["group"].unique()
group_colors   = {"ADULTO":"#E63946","ADOLESCENTE":"#2196F3","UNKNOWN":"#9E9E9E"}

fig, axes = plt.subplots(1, K, figsize=(4*K, 5), sharey=True)
for s, ax in enumerate(axes if K > 1 else [axes]):
    col = f"S{s}"
    for j, g in enumerate(groups_present):
        sub = fo_wide[fo_wide["group"]==g][col].dropna()
        bp  = ax.boxplot(sub, positions=[j], patch_artist=True,
                         widths=0.5, medianprops=dict(color="black",lw=2))
        bp["boxes"][0].set_facecolor(group_colors.get(g,"gray"))
        bp["boxes"][0].set_alpha(0.7)
        ax.scatter(np.random.normal(j,0.05,len(sub)), sub,
                   alpha=0.5, s=15, color=group_colors.get(g,"gray"), zorder=3)
    ax.set_title(f"S{s}  FO={df_fo[df_fo.state==col].FO.mean():.3f}",
                 fontweight="bold", color=STATE_COLORS[s])
    ax.set_xticks(range(len(groups_present)))
    ax.set_xticklabels(groups_present, fontsize=9)
    ax.axhline(1/K, color="gray", linestyle="--", alpha=0.4)
    if s == 0: ax.set_ylabel("Fractional Occupancy")

# Absorbentes
absorbing = summary.get("absorbing", [])
print(f"Sesiones absorbentes (FO>0.95): {len(absorbing)}")
if absorbing:
    df_abs = pd.DataFrame(absorbing)
    print(df_abs.groupby(["group","state"]).size().to_string())

plt.suptitle(f"FO por sujeto — {EXP_NAME}", fontsize=13, fontweight="bold")
plt.tight_layout(); save_fig("fo_by_subject"); plt.show()


---
## Balance Go vs NoGo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for col_i, cond in enumerate(["GO","NOGO"]):
    ax   = axes[col_i]
    sub  = df_fo[df_fo["condition"]==cond]
    means = sub.groupby("state")["FO"].mean()
    sems  = sub.groupby("state")["FO"].sem()
    x     = np.arange(K)
    ax.bar(x, [means.get(f"S{s}",0) for s in range(K)],
           yerr=[sems.get(f"S{s}",0) for s in range(K)],
           color=STATE_COLORS[:K], alpha=0.85, capsize=5, error_kw={"lw":2})
    ax.axhline(1/K, color="gray", linestyle="--", alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels([f"S{s}" for s in range(K)])
    n_cond = len(sub["subject_id"].unique())
    ax.set_title(f"FO media — {cond} (n={n_cond})", fontweight="bold")
    ax.set_ylabel("Fractional Occupancy")

plt.suptitle(f"Go vs NoGo — {EXP_NAME}", fontsize=13, fontweight="bold")
plt.tight_layout(); save_fig("fo_by_condition"); plt.show()


---
## Dwell times

In [ ]:
def compute_dwells(v, step_ms):
    dwells = {s: [] for s in range(K)}
    i, n = 0, len(v)
    while i < n:
        s = v[i]; j = i
        while j < n and v[j] == s: j += 1
        dwells[s].append((j-i)*step_ms)
        i = j
    return dwells

dwells = compute_dwells(viterbi, STEP_MS)

fig, axes = plt.subplots(1, K, figsize=(4*K, 4), sharey=False)
for s, ax in enumerate(axes if K > 1 else [axes]):
    d = np.array(dwells[s]) if dwells[s] else np.array([0.0])
    d_plot = d[d <= np.percentile(d, 95)] if len(d) > 1 else d
    ax.hist(d_plot, bins=40, color=STATE_COLORS[s], alpha=0.8, density=True)
    ax.axvline(np.median(d), color="black", lw=2, linestyle="-",
               label=f"med={np.median(d):.0f}ms")
    ax.axvline(STEP_MS/max(1-transmat[s,s],1e-9), color="red", lw=2, linestyle="--",
               label=f"analitico={STEP_MS/max(1-transmat[s,s],1e-9):.0f}ms")
    ax.set_title(f"S{s}", fontweight="bold", color=STATE_COLORS[s])
    ax.set_xlabel("Dwell time (ms)"); ax.legend(fontsize=8)

plt.suptitle(f"Dwell times — {EXP_NAME}", fontsize=13, fontweight="bold")
plt.tight_layout(); save_fig("dwell_times"); plt.show()

print("
Resumen dwell times:")
for s in range(K):
    d = np.array(dwells[s]) if dwells[s] else np.array([0.0])
    print(f"  S{s}: median={np.median(d):.0f}ms | mean={np.mean(d):.0f}ms | "
          f"p95={np.percentile(d,95):.0f}ms | n_runs={len(d):,}")


---
## Soft decoding

In [ ]:
confidence = np.max(posterior, axis=1)
H_norm     = -np.sum(posterior * np.log(posterior+1e-12), axis=1) / np.log(K)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(confidence, bins=60, color="#2196F3", alpha=0.85, density=True)
axes[0].axvline(np.mean(confidence), color="red", lw=2, linestyle="--",
                label=f"media={np.mean(confidence):.3f}")
axes[0].set_title("Confianza global", fontweight="bold")
axes[0].set_xlabel("Confianza"); axes[0].set_xlim(0,1); axes[0].legend()

axes[1].hist(H_norm, bins=60, color="#E63946", alpha=0.85, density=True)
axes[1].axvline(np.mean(H_norm), color="navy", lw=2, linestyle="--",
                label=f"media={np.mean(H_norm):.3f}")
axes[1].set_title("Entropia normalizada", fontweight="bold")
axes[1].set_xlabel("H norm"); axes[1].set_xlim(0,1); axes[1].legend()

mean_post = posterior.mean(axis=0)
axes[2].bar([f"S{s}" for s in range(K)], mean_post, color=STATE_COLORS[:K], alpha=0.85)
axes[2].axhline(1/K, color="gray", linestyle="--", alpha=0.6)
axes[2].set_title("Posterior medio por estado", fontweight="bold")
axes[2].set_ylabel("P(estado)")

plt.suptitle(f"Soft decoding — {EXP_NAME}", fontsize=13, fontweight="bold")
plt.tight_layout(); save_fig("soft_decoding"); plt.show()

interp = ("discreto" if np.mean(confidence)>0.85 else
          "metaestable" if np.mean(confidence)>0.65 else "continuo")
print(f"regimen: {interp} | confianza media={np.mean(confidence):.4f}")


---
## Secuencia temporal

In [ ]:
# Sujeto con FO mas balanceada (no absorbente)
abs_set = {a["subject_id"] for a in summary.get("absorbing", [])}
fo_balance = []
for m in subject_meta:
    if m["condition"] != "NOGO" or m["subject_id"] in abs_set:
        fo_balance.append(999)
        continue
    v_s = viterbi[m["start"]:m["end"]]
    fo_s = [np.mean(v_s==s) for s in range(K)]
    fo_balance.append(np.std(fo_s))

best_idx = int(np.argmin(fo_balance))
best_sub = subject_meta[best_idx]
print(f"sujeto seleccionado: {best_sub[chr(115)+chr(117)+chr(98)+chr(106)+chr(101)+chr(99)+chr(116)+chr(95)+chr(105)+chr(100)]} ({best_sub[chr(99)+chr(111)+chr(110)+chr(100)+chr(105)+chr(116)+chr(105)+chr(111)+chr(110)]}, {best_sub[chr(103)+chr(114)+chr(111)+chr(117)+chr(112)]})")

def plot_timeseries(sub, title=""):
    v_s = viterbi[sub["start"]:sub["end"]]
    p_s = posterior[sub["start"]:sub["end"]]
    t_s = np.arange(len(v_s)) * STEP_MS / 1000
    fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
    for s in range(K):
        axes[0].fill_between(t_s, s-0.4, s+0.4, where=(v_s==s),
                             color=STATE_COLORS[s], alpha=0.85)
    axes[0].set_yticks(range(K)); axes[0].set_yticklabels([f"S{s}" for s in range(K)])
    axes[0].set_ylabel("Estado (Viterbi)")
    axes[0].set_title(f"{sub[chr(115)+chr(117)+chr(98)+chr(106)+chr(101)+chr(99)+chr(116)+chr(95)+chr(105)+chr(100)]} ({sub[chr(99)+chr(111)+chr(110)+chr(100)+chr(105)+chr(116)+chr(105)+chr(111)+chr(110)]}){title}", fontweight="bold")
    for s in range(K):
        axes[1].plot(t_s, p_s[:,s], color=STATE_COLORS[s], lw=1.5, alpha=0.85, label=f"S{s}")
    axes[1].set_ylabel("P(estado | datos)"); axes[1].set_xlabel("Tiempo (s)")
    axes[1].set_ylim(0,1); axes[1].legend(loc="upper right")
    plt.suptitle(f"Dinamica temporal — {EXP_NAME}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    return fig

fig = plot_timeseries(best_sub, " — NOGO"); save_fig("timeseries_nogo"); plt.show()

# Par GO del mismo sujeto
target = best_sub["subject_id"]
go_sub = next((m for m in subject_meta
               if m["subject_id"]==target and m["condition"]=="GO"), None)
if go_sub:
    fig = plot_timeseries(go_sub, " — GO"); save_fig("timeseries_go"); plt.show()

# Absorbente
if summary.get("absorbing"):
    abs_id = summary["absorbing"][0]["subject_id"]
    abs_sub = next((m for m in subject_meta if m["subject_id"]==abs_id), None)
    if abs_sub:
        v_a = viterbi[abs_sub["start"]:abs_sub["end"]]
        t_a = np.arange(len(v_a)) * STEP_MS / 1000
        fig, ax = plt.subplots(figsize=(15, 2.5))
        for s in range(K):
            ax.fill_between(t_a, s-0.4, s+0.4, where=(v_a==s),
                            color=STATE_COLORS[s], alpha=0.85)
        ax.set_yticks(range(K)); ax.set_yticklabels([f"S{s}" for s in range(K)])
        ax.set_xlabel("Tiempo (s)")
        ax.set_title(f"Absorbente — {abs_id}", fontweight="bold", color="red")
        plt.tight_layout(); save_fig("timeseries_absorbing"); plt.show()


---
## Diagnostico

In [ ]:
print("="*55)
print(f"DIAGNOSTICO — {EXP_NAME}")
print("="*55)

for k, v in summary["checks"].items():
    print(f"  {chr(79)+chr(75) if v else chr(70)+chr(65)+chr(73)+chr(76)}  {k}")

print(f"
Score : {summary[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)]}/4")
print(f"Veredicto: {summary[chr(118)+chr(101)+chr(114)+chr(100)+chr(105)+chr(99)+chr(116)]}")
print(f"Absorbentes: {summary[chr(110)+chr(95)+chr(97)+chr(98)+chr(115)+chr(111)+chr(114)+chr(98)+chr(105)+chr(110)+chr(103)]}/{len(subject_meta)} sesiones")

tr = sum(1 for i in range(len(viterbi)-1) if viterbi[i]!=viterbi[i+1])
print(f"Transition rate: {tr/(len(viterbi)*STEP_MS/1000):.3f} trans/seg")
